In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
df = pd.read_csv("Deliverable2_cleaned.csv")
df.sample(5)

,X,Y,NCESSCH,STABR,ST_LEAID,LEA_NAME,SCH_NAME,LSTREET1,LCITY,LSTATE,...,STUTERATIO,AM,AS,BL,HP,HI,TR,WH,TOTAL_RACE_CONS,TOTAL_GRADE_CONS
90576,-77.715568,38.737523,510132002409,VA,VA-030,Fauquier County Public Schools,AUBURN MIDDLE,7270 Riley Rd.,Warrenton,VA,...,12.96,0.361011,0.902527,4.512635,0.000000,10.830325,6.137184,77.256318,True,True
19651,-81.629500,30.265300,120048000678,FL,FL-16,DUVAL,SAN JOSE ELEMENTARY SCHOOL,5805 SAINT AUGUSTINE RD,JACKSONVILLE,FL,...,15.27,0.898588,12.451861,19.255456,0.898588,46.341463,2.695764,17.458280,True,True
81414,-96.072635,31.470117,481199000701,TX,TX-145901,BUFFALO ISD,BUFFALO H S,1724 N BUFFALO AVE,BUFFALO,TX,...,11.72,0.316456,1.265823,3.164557,0.000000,45.253165,2.215190,47.784810,True,True
65677,-81.526961,41.429040,390027904819,OH,OH-151209,Randall Park High School,Randall Park High School,4836 Northfield Rd,North Randall,OH,...,28.80,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,True,True
6413,-118.371282,34.022000,60166813899,CA,CA-0134148,The City District,The City,5753 Obama Blvd.,Los Angeles,CA,...,24.59,1.333333,1.333333,49.333333,0.000000,39.333333,2.666667,6.000000,True,True


In [16]:
output_folder = "eda_plots"
os.makedirs(output_folder, exist_ok=True)

target_col = "STUTERATIO"

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include='number').columns
categorical_cols = df.select_dtypes(exclude='number').columns


In [18]:
df["STUTERATIO"].median()

14.64

## Checking Columns Relationship with STUTERATIO

In [17]:
# -----------------------------
# 1️⃣ Scatter plots: STUTERATIO vs other numeric columns
# -----------------------------
for col in numeric_cols:
    if col == target_col:
        continue
    
    plt.figure(figsize=(8,5))
    sns.scatterplot(data=df, x=col, y=target_col, s=80)
    plt.title(f'{target_col} vs {col}')
    plt.xlabel(col)
    plt.ylabel(target_col)
    plt.savefig(f"{output_folder}/{target_col}_vs_{col}_scatter.png", bbox_inches='tight')
    plt.close()


In [14]:
# List of specific categorical columns you want to analyze
target_categories = [
    'LSTATE', 'VIRTUAL', 'GSLO', 'GSHI', 
    'SCHOOL_LEVEL', 'SCHOOL_TYPE_TEXT', 'SY_STATUS_TEXT', 'ULOCALE'
]

# -----------------------------
# 2️⃣ Box Plots: Categorical Interaction with STUTERATIO
# -----------------------------
for col in target_categories:
    if col in df.columns:
        plt.figure(figsize=(12, 6))
        
        order = df.groupby(col)[target_col].median().sort_values().index
        
        # Updated syntax to satisfy the FutureWarning
        sns.boxplot(
            data=df, 
            x=col, 
            y=target_col, 
            order=order,
            hue=col,          # Map the x-axis to the color (hue)
            legend=False,     # Hide the legend (since x-axis labels already explain the boxes)
            palette='viridis'
        )
        
        plt.title(f'Distribution of {target_col} by {col}', fontsize=14)
        plt.xticks(rotation=45, ha='right')
        plt.savefig(f"{output_folder}/{target_col}_vs_{col}_boxplot.png", bbox_inches='tight')
        plt.close()

## Conclusion
1. SCHOOL_TYPE_TEXT\
   Special education school has lower STUTERATIO
2. SCHOOL_LEVEL\
   Ungraded has lower STUTERATIO
3. ULOCALE\
   Suburban + City has higher than average, whereas Rural has lower than average
4. LSTATE\
   Who are significantly lower than average and vice versa.
5. AS\
   When AS student ratio over 35, the STUTERATIO become stable than average.
6. WH\
   WH student ratio does not affect the STUTERATIO. 

## Statistical Analysis
### 1. SCHOOL_TYPE_TEXT
Special education school has lower STUTERATIO
#### Hypotheses

##### Null Hypothesis (H₀)

μ_special ≥ μ_population  

The mean student–teacher ratio of special education schools is equal to or greater than the population mean.

##### Alternative Hypothesis (H₁)

μ_special < μ_population  

The mean student–teacher ratio of special education schools is lower than the population mean.

##### Significance Level

α = 0.05

#### Test Type

One-sample, one-tailed t-test

In [34]:
import numpy as np
from scipy import stats

# 1. Define population and sample
mu_population = df["STUTERATIO"].mean()
special = df[df["SCHOOL_TYPE_TEXT"] == "Special Education School"]["STUTERATIO"].dropna()

n = len(special)
sample_mean = special.mean()
sample_std = special.std()

print(f"Population Mean: {mu_population:.2f}")
print(f"Sample size (special schools): {n}")
print(f"Mean (special schools): {sample_mean:.2f}")

# 2. Calculate 95% Confidence Interval for the sample mean
# Formula: mean +/- t * (std / sqrt(n))
ci_lower, ci_upper = stats.t.interval(
    0.95, 
    df=n-1, 
    loc=sample_mean, 
    scale=stats.sem(special) # stats.sem calculates std/sqrt(n)
)

print(f"95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 3. One-sample t-test (Left-tailed)
t_stat, p_two_tailed = stats.ttest_1samp(special, mu_population)

# Convert to one-tailed p-value (left-tailed)
p_one_tailed = p_two_tailed / 2
if t_stat > 0:
    p_one_tailed = 1 - p_one_tailed

print(f"T-statistic: {t_stat:.4f}")
print(f"One-tailed p-value: {p_one_tailed:.4e}")

# 4. Conclusion
alpha = 0.05
print("-" * 30)
if p_one_tailed < alpha:
    print("RESULT: Reject H0")
    print("Conclusion: Special education schools have a significantly lower STUTERATIO.")
    print(f"The 95% CI [{ci_lower:.2f}, {ci_upper:.2f}] is entirely below the population mean of {mu_population:.2f}.")
else:
    print("RESULT: Fail to reject H0")
    print("Conclusion: No significant evidence that special education schools have a lower STUTERATIO.")

Population Mean: 15.14
Sample size (special schools): 1599
Mean (special schools): 9.31
95% Confidence Interval: [8.90, 9.72]
T-statistic: -27.6258
One-tailed p-value: 5.8988e-138
------------------------------
RESULT: Reject H0
Conclusion: Special education schools have a significantly lower STUTERATIO.
The 95% CI [8.90, 9.72] is entirely below the population mean of 15.14.


### 2. SCHOOL_LEVEL
Ungraded has lower STUTERATIO

#### Hypotheses

##### Null Hypothesis (H₀)

μ_ungraded ≥ μ_population  

The mean student–teacher ratio of ungraded schools is equal to or greater than the population mean.

##### Alternative Hypothesis (H₁)

μ_ungraded < μ_population  

The mean student–teacher ratio of ungraded schools is lower than the population mean.

##### Significance Level

α = 0.05

#### Test Type

One-sample, one-tailed t-test

In [35]:
import numpy as np
from scipy import stats

# 1. Population mean
mu_population = df["STUTERATIO"].mean()

# 2. Extract and clean Ungraded schools data
ungraded = df[df["SCHOOL_LEVEL"] == "Ungraded"]["STUTERATIO"].dropna()
n = len(ungraded)
sample_mean = ungraded.mean()

print(f"Population Mean: {mu_population:.2f}")
print(f"Sample size (ungraded schools): {n}")
print(f"Mean (ungraded schools): {sample_mean:.2f}")

# 3. Calculate 95% Confidence Interval
# Standard Error of the Mean (SEM) = std / sqrt(n)
ci_lower, ci_upper = stats.t.interval(
    0.95, 
    df=n-1, 
    loc=sample_mean, 
    scale=stats.sem(ungraded)
)

print(f"95% Confidence Interval for Mean: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 4. One-sample t-test (Left-tailed)
t_stat, p_two_tailed = stats.ttest_1samp(ungraded, mu_population)

# Convert to one-tailed p-value (targeting 'lower')
p_one_tailed = p_two_tailed / 2
if t_stat > 0:
    p_one_tailed = 1 - p_one_tailed

print(f"T-statistic: {t_stat:.4f}")
print(f"One-tailed p-value: {p_one_tailed:.4e}")

# 5. Final Conclusion
alpha = 0.05
print("-" * 30)
if p_one_tailed < alpha:
    print("RESULT: Reject H0")
    print("Conclusion: Ungraded schools have a significantly lower STUTERATIO.")
    print(f"The confidence interval [{ci_lower:.2f}, {ci_upper:.2f}] confirms the mean is strictly below {mu_population:.2f}.")
else:
    print("RESULT: Fail to reject H0")
    print("Conclusion: No significant evidence that ungraded schools have a lower STUTERATIO.")

Population Mean: 15.14
Sample size (ungraded schools): 134
Mean (ungraded schools): 2.16
95% Confidence Interval for Mean: [1.51, 2.80]
T-statistic: -39.9194
One-tailed p-value: 3.3032e-76
------------------------------
RESULT: Reject H0
Conclusion: Ungraded schools have a significantly lower STUTERATIO.
The confidence interval [1.51, 2.80] confirms the mean is strictly below 15.14.


### 3. ULOCALE
Suburban + City has higher than average, whereas Rural has lower than average

#### Hypotheses (City + Suburb)

##### Null Hypothesis (H₀)

μ_city_suburb ≤ μ_population  

The mean student–teacher ratio of City and Suburban schools is equal to or less than the population mean.

##### Alternative Hypothesis (H₁)

μ_city_suburb > μ_population  

The mean student–teacher ratio of City and Suburban schools is greater than the population mean.

##### Significance Level

α = 0.05

#### Test Type

One-sample, one-tailed t-test (right-tailed)

---

#### Hypotheses (Rural)

##### Null Hypothesis (H₀)

μ_rural ≥ μ_population  

The mean student–teacher ratio of Rural schools is equal to or greater than the population mean.

##### Alternative Hypothesis (H₁)

μ_rural < μ_population  

The mean student–teacher ratio of Rural schools is lower than the population mean.

##### Significance Level

α = 0.05

#### Test Type

One-sample, one-tailed t-test (left-tailed)

In [36]:
import numpy as np
from scipy import stats

# 1. Population mean
mu_population = df["STUTERATIO"].mean()

# 2. Extract and clean City (1) and Suburb (2) data
# We use dropna() to ensure the t-test and SEM calculations are accurate
city_suburb = df[df["ULOCALE"].astype(str).str.startswith(("1", "2"))]["STUTERATIO"].dropna()

n = len(city_suburb)
sample_mean = city_suburb.mean()

print(f"Population Mean: {mu_population:.2f}")
print(f"Sample size (City + Suburb): {n}")
print(f"Mean (City + Suburb): {sample_mean:.2f}")

# 3. Calculate 95% Confidence Interval for the mean
# Standard Error (SEM) = std / sqrt(n)
ci_lower, ci_upper = stats.t.interval(
    0.95, 
    df=n-1, 
    loc=sample_mean, 
    scale=stats.sem(city_suburb)
)

print(f"95% Confidence Interval for Mean: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 4. One-sample t-test (Right-tailed)
t_stat, p_two_tailed = stats.ttest_1samp(city_suburb, mu_population)

# Convert to one-tailed p-value (targeting 'higher')
p_one_tailed = p_two_tailed / 2

# Adjust direction for right-tailed test: 
# If t_stat is negative, it means the sample mean is LESS than the population mean
if t_stat < 0:
    p_one_tailed = 1 - p_one_tailed

print(f"T-statistic: {t_stat:.4f}")
print(f"One-tailed p-value: {p_one_tailed:.4e}")

# 5. Final Conclusion
alpha = 0.05
print("-" * 30)
if p_one_tailed < alpha:
    print("RESULT: Reject H0")
    print("Conclusion: City + Suburb schools have a significantly higher STUTERATIO.")
    print(f"The 95% CI [{ci_lower:.2f}, {ci_upper:.2f}] is entirely above the population mean of {mu_population:.2f}.")
else:
    print("RESULT: Fail to reject H0")
    print("Conclusion: No significant evidence that City + Suburb schools have a higher STUTERATIO.")

Population Mean: 15.14
Sample size (City + Suburb): 57823
Mean (City + Suburb): 15.72
95% Confidence Interval for Mean: [15.67, 15.76]
T-statistic: 23.5175
One-tailed p-value: 5.0525e-122
------------------------------
RESULT: Reject H0
Conclusion: City + Suburb schools have a significantly higher STUTERATIO.
The 95% CI [15.67, 15.76] is entirely above the population mean of 15.14.


In [37]:
import numpy as np
from scipy import stats

# 1. Population mean
mu_population = df["STUTERATIO"].mean()

# 2. Extract and clean City (1) and Suburb (2) data
# We use dropna() to ensure the t-test and SEM calculations are accurate
city_suburb = df[df["ULOCALE"].astype(str).str.startswith(("1", "2"))]["STUTERATIO"].dropna()

n = len(city_suburb)
sample_mean = city_suburb.mean()

print(f"Population Mean: {mu_population:.2f}")
print(f"Sample size (City + Suburb): {n}")
print(f"Mean (City + Suburb): {sample_mean:.2f}")

# 3. Calculate 95% Confidence Interval for the mean
# Standard Error (SEM) = std / sqrt(n)
ci_lower, ci_upper = stats.t.interval(
    0.95, 
    df=n-1, 
    loc=sample_mean, 
    scale=stats.sem(city_suburb)
)

print(f"95% Confidence Interval for Mean: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 4. One-sample t-test (Right-tailed)
t_stat, p_two_tailed = stats.ttest_1samp(city_suburb, mu_population)

# Convert to one-tailed p-value (targeting 'higher')
p_one_tailed = p_two_tailed / 2

# Adjust direction for right-tailed test: 
# If t_stat is negative, it means the sample mean is LESS than the population mean
if t_stat < 0:
    p_one_tailed = 1 - p_one_tailed

print(f"T-statistic: {t_stat:.4f}")
print(f"One-tailed p-value: {p_one_tailed:.4e}")

# 5. Final Conclusion
alpha = 0.05
print("-" * 30)
if p_one_tailed < alpha:
    print("RESULT: Reject H0")
    print("Conclusion: City + Suburb schools have a significantly higher STUTERATIO.")
    print(f"The 95% CI [{ci_lower:.2f}, {ci_upper:.2f}] is entirely above the population mean of {mu_population:.2f}.")
else:
    print("RESULT: Fail to reject H0")
    print("Conclusion: No significant evidence that City + Suburb schools have a higher STUTERATIO.")

Population Mean: 15.14
Sample size (City + Suburb): 57823
Mean (City + Suburb): 15.72
95% Confidence Interval for Mean: [15.67, 15.76]
T-statistic: 23.5175
One-tailed p-value: 5.0525e-122
------------------------------
RESULT: Reject H0
Conclusion: City + Suburb schools have a significantly higher STUTERATIO.
The 95% CI [15.67, 15.76] is entirely above the population mean of 15.14.


## 4. LSTATE
Which states have significantly higher or lower STUTERATIO than the population average?

### Hypotheses (For Each State)

Let:

μ_state = Mean STUTERATIO of a specific state  
μ_population = Mean STUTERATIO of all schools  

#### Null Hypothesis (H₀)

μ_state = μ_population  

The state's mean student–teacher ratio is equal to the population mean.

#### Alternative Hypothesis (H₁)

μ_state ≠ μ_population  

The state's mean student–teacher ratio is different from the population mean.

Significance Level: α = 0.05  
Test Type: One-sample t-test (two-tailed)


In [33]:
import numpy as np
from scipy import stats

# 1. Define population parameters
mu_population = df["STUTERATIO"].mean()
alpha = 0.05

significant_states = []

# 2. Iterate through states
for state in df["LSTATE"].unique():
    state_data = df[df["LSTATE"] == state]["STUTERATIO"].dropna()
    
    n = len(state_data)
    if n < 10:  # Stay with 10 for statistical stability
        continue
        
    # Perform one-sample t-test
    t_stat, p_value = stats.ttest_1samp(state_data, mu_population)
    
    # 3. Check for statistical significance
    if p_value < alpha:
        state_mean = state_data.mean()
        state_std = state_data.std()
        
        # Calculate 95% Confidence Interval for the State Mean
        # stats.t.interval gives the range (lower, upper)
        ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=state_mean, scale=state_std/np.sqrt(n))
        
        direction = "HIGHER 📈" if state_mean > mu_population else "LOWER 📉"
        
        significant_states.append({
            "State": state,
            "Direction": direction,
            "Mean": round(state_mean, 2),
            "CI": f"[{ci_lower:.2f}, {ci_upper:.2f}]",
            "P-Value": f"{p_value:.4e}" # Using scientific notation for very small p-values
        })

# 4. Print Results
print(f"Population Average STUTERATIO: {mu_population:.2f}\n")
print(f"{'State':<6} | {'Status':<10} | {'Mean':<6} | {'95% Conf. Interval':<16} | {'P-Value'}")
print("-" * 75)

for s in significant_states:
    print(f"{s['State']:<6} | {s['Direction']:<10} | {s['Mean']:<6} | {s['CI']:<16} | {s['P-Value']}")

Population Average STUTERATIO: 15.14

State  | Status     | Mean   | 95% Conf. Interval | P-Value
---------------------------------------------------------------------------
AL     | HIGHER 📈   | 17.43  | [17.26, 17.60]   | 8.4183e-130
AZ     | HIGHER 📈   | 17.33  | [17.13, 17.53]   | 3.2703e-96
AR     | LOWER 📉    | 13.25  | [12.98, 13.53]   | 9.0672e-39
CA     | HIGHER 📈   | 21.16  | [21.06, 21.27]   | 0.0000e+00
CO     | HIGHER 📈   | 16.09  | [15.82, 16.37]   | 2.4673e-11
CT     | LOWER 📉    | 11.99  | [11.79, 12.18]   | 2.0012e-154
DE     | LOWER 📉    | 13.56  | [13.10, 14.01]   | 7.6345e-11
DC     | LOWER 📉    | 11.62  | [10.92, 12.32]   | 8.8032e-20
MD     | LOWER 📉    | 13.99  | [13.80, 14.18]   | 2.0387e-30
FL     | HIGHER 📈   | 17.38  | [17.18, 17.58]   | 8.2312e-99
GA     | LOWER 📉    | 14.34  | [14.21, 14.47]   | 1.2077e-32
HI     | LOWER 📉    | 14.23  | [13.94, 14.53]   | 7.2768e-09
ID     | HIGHER 📈   | 16.6   | [16.21, 17.00]   | 1.2321e-12
OR     | HIGHER 📈   | 17.25  | 

## 5. AS Student Ratio Stability Analysis

**Insight:** Schools where the AS student ratio exceeds 35% exhibit a `STUTERATIO` that is significantly more stable (lower variance) than the general population.

### Hypotheses

To validate this "stability," we compare the **Variances ($\sigma^2$)** of the two groups using an **F-Test for Equality of Variances**.

Let:

- $\sigma^2_{AS>35}$ = Variance of `STUTERATIO` for schools with AS ratio > 35%
    
- $\sigma^2_{pop}$ = Variance of `STUTERATIO` for the entire population (All schools)
    

#### Null Hypothesis ($H_0$)

$\sigma^2_{AS>35} \geq \sigma^2_{pop}$

_(The AS group is as volatile or more volatile than the general population.)_

#### Alternative Hypothesis ($H_1$)

$\sigma^2_{AS>35} < \sigma^2_{pop}$

_(The AS group is significantly more stable/less volatile than the general population.)_

- **Significance Level:** $\alpha = 0.05$
    
- **Test Type:** F-Test (Left-tailed)

In [38]:
import numpy as np
from scipy import stats

# 1. Prepare the Data
high_as_group = df[df['AS'] > 35]['STUTERATIO'].dropna()
whole_pop = df['STUTERATIO'].dropna()

# 2. Calculate Variances (ddof=1 for sample variance)
var_high_as = np.var(high_as_group, ddof=1)
var_pop = np.var(whole_pop, ddof=1)
n = len(high_as_group)
df_high = n - 1

# 3. Calculate 95% Confidence Interval for the Variance (High-AS Group)
# Formula: [(n-1)s^2 / chi2.upper, (n-1)s^2 / chi2.lower]
conf_level = 0.95
chi2_lower = stats.chi2.ppf((1 - conf_level) / 2, df_high)
chi2_upper = stats.chi2.ppf((1 + conf_level) / 2, df_high)

ci_var_lower = (df_high * var_high_as) / chi2_upper
ci_var_upper = (df_high * var_high_as) / chi2_lower

# 4. Perform the F-Test
f_stat = var_high_as / var_pop
df_pop = len(whole_pop) - 1
p_value = stats.f.cdf(f_stat, df_high, df_pop)

# 5. Results Output
print(f"--- Hypothesis Testing: AS Ratio Stability ---")
print(f"Variance (AS > 35%): {var_high_as:.4f}")
print(f"95% CI for Variance: [{ci_var_lower:.4f}, {ci_var_upper:.4f}]")
print(f"Variance (Whole Pop): {var_pop:.4f}")
print("-" * 45)
print(f"F-statistic:         {f_stat:.4f}")
print(f"P-value:             {p_value:.4f}")
print("-" * 45)

if p_value < 0.05:
    print("RESULT: REJECT the Null Hypothesis (H0).")
    print("Conclusion: The STUTERATIO is significantly more stable in the high-AS group.")
    print(f"Proof: The entire variance CI is below the population variance of {var_pop:.4f}.")
else:
    print("RESULT: FAIL TO REJECT the Null Hypothesis (H0).")
    print("Conclusion: No statistical evidence of higher stability.")

--- Hypothesis Testing: AS Ratio Stability ---
Variance (AS > 35%): 31.7295
95% CI for Variance: [29.8143, 33.8364]
Variance (Whole Pop): 33.6500
---------------------------------------------
F-statistic:         0.9429
P-value:             0.0379
---------------------------------------------
RESULT: REJECT the Null Hypothesis (H0).
Conclusion: The STUTERATIO is significantly more stable in the high-AS group.
Proof: The entire variance CI is below the population variance of 33.6500.


###### **Small Conclusion**
"We performed an F-test to determine if schools with an Asian student ratio > 35% exhibit more stable Student-Teacher Ratios than the general population. With a variance of 31.73 compared to the population's 33.65, the resulting p-value of 0.0379 allows us to reject the null hypothesis. We can conclude with 95% confidence that the high-AS group is significantly more stable, with its true variance likely falling between 29.81 and 33.84."

## 6. WH Student Ratio

WH student ratio does not affect STUTERATIO.

### Hypotheses

We test whether schools with high WH ratio differ from population mean.

Let:

μ_WH>threshold = Mean STUTERATIO for high WH schools  

#### Null Hypothesis (H₀)

μ_WH = μ_population  

#### Alternative Hypothesis (H₁)

μ_WH ≠ μ_population  

Significance Level: α = 0.05  
Test Type: One-sample t-test (two-tailed)

In [39]:
import numpy as np
from scipy import stats

# 1. Population Parameters
mu_population = df["STUTERATIO"].mean()
alpha = 0.05

# 2. Extract Target Group (WH > 50%)
wh_high = df[df["WH"] > 50]["STUTERATIO"].dropna()
n = len(wh_high)
sample_mean = wh_high.mean()

print(f"Population Mean: {mu_population:.2f}")
print(f"Sample size (WH > 50%): {n}")
print(f"Sample Mean: {sample_mean:.2f}")

# 3. Calculate 95% Confidence Interval
# Standard Error (SEM) = std / sqrt(n)
ci_lower, ci_upper = stats.t.interval(
    0.95, 
    df=n-1, 
    loc=sample_mean, 
    scale=stats.sem(wh_high)
)

print(f"95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 4. One-sample t-test (Two-tailed)
t_stat, p_value = stats.ttest_1samp(wh_high, mu_population)

print(f"T-statistic: {t_stat:.4f}")
print(f"Two-tailed p-value: {p_value:.4e}")

# 5. Final Conclusion
print("-" * 30)
if p_value < alpha:
    direction = "higher" if sample_mean > mu_population else "lower"
    print(f"RESULT: Reject H0")
    print(f"Conclusion: WH ratio (>50%) significantly affects STUTERATIO (it is {direction}).")
    print(f"The 95% CI { [round(ci_lower, 2), round(ci_upper, 2)] } does not contain the population mean {mu_population:.2f}.")
else:
    print("RESULT: Fail to reject H0")
    print("Conclusion: WH ratio (>50%) does not significantly affect STUTERATIO.")

Population Mean: 15.14
Sample size (WH > 50%): 48309
Sample Mean: 14.72
95% Confidence Interval: [14.67, 14.76]
T-statistic: -17.7492
Two-tailed p-value: 2.9229e-70
------------------------------
RESULT: Reject H0
Conclusion: WH ratio (>50%) significantly affects STUTERATIO (it is lower).
The 95% CI [np.float64(14.67), np.float64(14.76)] does not contain the population mean 15.14.
